# Chapter 8 — Measuring similarity · Annotated solution

*Maths for AI, Step by Step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch08_corrige.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

Each answer first presents the calculation by hand, as in the book's solutions, then checks it in Python.

**Contents.** Data · The computations of lessons 8.1 to 8.3 · Exercises 8.1 to 8.6

**Before opening Jupyter:** the exercises in this chapter are done first on paper, with a calculator. The cells below are there to **check** your calculations. As in the book, means and standard deviations divide by $n$ and are computed on the six reference sales, A to F.

**Variable names.** The code uses the book's English names: `floor_area` (floor area, in m²), `distance` (distance to the center, in km), `price` (price, in k€), `home` (name of the home).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The book's colors: reference homes in blue, selected neighbors circled in orange, distances in magenta
BLUE, ORANGE, MAGENTA = '#2a78d6', '#eb6834', '#c2185b'

## Data

The identifiers D2.1 and D2.E2 are those of the book and of the CSV files. The columns are renamed in English when the data are read; the correspondence is given in a comment.

In [ ]:
# Chapter 8 data, copied from the book's CSV files (do not modify)

# D2.1 — the seven homes; CSV columns renamed in English:
#   home = logement, floor_area = surface_m2 (m²), distance = distance_km (km), price = prix_kEUR (k€)
homes = pd.DataFrame({
    'home':       ['A', 'B', 'C', 'D', 'E', 'F', 'G'],
    'floor_area': [32, 45, 54, 70, 85, 98, 62],
    'distance':   [3.0, 1.5, 2.5, 1.0, 2.5, 0.5, 2.5],
    'price':      [118, 172, 185, 245, 279, 335, 209],
})
reference = homes[homes['home'] != 'G'].reset_index(drop=True)   # the six reference sales, A to F
X = reference[['floor_area', 'distance']].to_numpy(dtype=float)  # one row per sale: (m², km)
price = reference['price'].to_numpy(dtype=float)                 # prices of the reference sales, in k€
G = np.array([62, 2.5])                                          # home G: 62 m², 2.5 km (sold for 209 k€)

# D2.E2 — home I (exercise 8.2)
I = np.array([75, 3.5])   # 75 m², 3.5 km

# Examples from the book (lessons 8.1 and 8.3): city map in km, messages, made-up embeddings
school, G_map = np.array([4.5, 6]), np.array([1.5, 2])            # the school and G on the map, in km
R1, R2, P = np.array([3, 1]), np.array([6, 2]), np.array([1, 3])  # (“meeting”, “offer”)
embeddings = {'meeting': np.array([4, 3]), 'assembly': np.array([3, 4]),
              'discount': np.array([-3, 4]), 'promotion': np.array([-4, 3])}
# keys: the four words of table 8.3 (meeting, assembly, discount, promotion)

### Data variables

Code names are in English. For the columns, the last column gives the name used in the book's CSV files.

| Name | Meaning | Unit | CSV column |
|:--|:--|:--|:--|
| `home` | home identifier (A, B…) | — | `logement` |
| `floor_area` | living area of the home | m² | `surface_m2` |
| `price` | sale price | k€ | `prix_kEUR` |
| `distance` | distance to the city center | km | `distance_km` |
| `distance_m` | distance to the city center | m | — |
| `message` | text identifier (T1 to T4) | — | — |
| `meeting_count` | number of occurrences of the word “meeting” | occurrences | — |
| `offer_count` | number of occurrences of the word “offer” | occurrences | — |
| `volunteer_count` | number of occurrences of the word “volunteer” | occurrences | — |

In [ ]:
# Provided tools: you do not need to modify them

def nearest(dists, k=3):
    """The k nearest reference sales (distances given in the order A to F) and their mean price."""
    order = np.argsort(dists, kind='stable')[:k]
    return list(reference['home'].iloc[order]), float(price[order].mean())


def show_table(dists, label):
    """Table of the distances to a home, from shortest to longest."""
    t = pd.DataFrame({'home': reference['home'], label: np.round(dists, 3), 'price': price})
    print(t.sort_values(label, kind='stable').to_string(index=False))

## The computations of the lessons, in Python

These cells redo the computations of lessons 8.1 to 8.3. Run them and compare with the book.

In [ ]:
# Lesson 8.1 — norm and distance (equation 8.1)
trip = school - G_map                       # displacement from G to the school, in km
print(f"displacement {trip}, distance {np.sqrt((trip ** 2).sum()):g} km")
print(f"from the center to G: {np.linalg.norm(G_map):g} km")             # the distance to the center in table 2.1
print(f"norm of (2, 3, 6): {np.linalg.norm([2, 3, 6]):g}")      # Pythagoras twice, or a single calculation
C_home = X[2]                                                       # C: 54 m², 2.5 km
print(f"distance from C to G: {np.linalg.norm(C_home - G):g} (mixed units: m² and km)")

In [ ]:
# Lesson 8.2 — first attempt: raw distances to G, then the distance to the center in meters
raw_G = np.sqrt(((X - G) ** 2).sum(axis=1))
show_table(raw_G, 'raw_distance')
n_km, e_km = nearest(raw_G)
print(f"3 nearest neighbors (raw, km): {n_km}, estimate {e_km:.1f} k€")
X_m, G_m = X * [1, 1000], G * [1, 1000]                             # distance to the center in meters
n_m, e_m = nearest(np.linalg.norm(X_m - G_m, axis=1))
print(f"3 nearest neighbors (raw, m):  {n_m}, estimate {e_m:.0f} k€")

In [ ]:
# Lesson 8.2 — second attempt: standardize with the reference sales (equation 8.2)
mu, sigma = X.mean(axis=0), X.std(axis=0)                           # division by n (ddof=0)
print(f"floor area: mean {mu[0]:.0f} m², standard deviation {sigma[0]:.2f} m²")
print(f"distance: mean {mu[1]:.3f} km, standard deviation {sigma[1]:.3f} km (√29/6 = {np.sqrt(29) / 6:.3f})")
Z, z_G = (X - mu) / sigma, (G - mu) / sigma
print('z of G:', np.round(z_G, 2))
std_G = np.linalg.norm(Z - z_G, axis=1)
show_table(std_G, 'std_distance')
neighbours, estimate = nearest(std_G)
print(f"3 nearest neighbors (standardized): {neighbours}, estimate {estimate:.0f} k€ (actual price: 209 k€)")

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(Z[:, 0], Z[:, 1], color=BLUE, s=40)
ax.scatter(*z_G, marker='D', color='black', s=40)
for h, (a, b) in zip(reference['home'], Z):
    ax.annotate(h, (a, b), xytext=(5, 4), textcoords='offset points')
ax.annotate('G', z_G, xytext=(5, 4), textcoords='offset points')
idx = np.argsort(std_G, kind='stable')[:3]
ax.scatter(Z[idx, 0], Z[idx, 1], s=220, facecolors='none', edgecolors=ORANGE, linewidths=1.8)
ax.add_patch(plt.Circle(z_G, std_G[idx[-1]], fill=False, linestyle='--', color='grey'))
ax.set_aspect('equal'); ax.set_xlabel('z floor area'); ax.set_ylabel('z distance'); ax.grid(alpha=0.4)
plt.show()

In [ ]:
# Lesson 8.3 — distance or cosine? (equation 8.3)
def cos_lesson(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

print(f"distances: R1–R2 {np.linalg.norm(R1 - R2):.2f}, R1–P {np.linalg.norm(R1 - P):.2f}")
print(f"cosines:   R1–R2 {cos_lesson(R1, R2):.2f}, R1–P {cos_lesson(R1, P):.2f}, "
      f"angle R1–P {np.degrees(np.arccos(cos_lesson(R1, P))):.0f}°")

# Preview: the embeddings of table 8.3 (all of norm 5)
for a, b in [('meeting', 'assembly'), ('discount', 'promotion'), ('meeting', 'discount'), ('meeting', 'promotion')]:
    print(f"cos({a}, {b}) = {cos_lesson(embeddings[a], embeddings[b]):.2f}")
message = (embeddings['meeting'] + embeddings['assembly']) / 2       # the message “meeting, assembly”
print(f"message {message}: cos with meeting {cos_lesson(message, embeddings['meeting']):.2f}, "
      f"with promotion {cos_lesson(message, embeddings['promotion']):.2f}")

## Exercise 8.1 — Distances on the city map

*Concepts: norm and distance (lesson 8.1, equation 8.1).* On the map in figure 8.1, in kilometers, the swimming pool is at the point $(7.5, 10)$ and the station at the point $(-3.5, 5)$; G is at the point $(1.5, 2)$.

### a) to e) The calculation by hand

a) $\sqrt{36 + 64} = \sqrt{100} = 10$.

b) From G to the swimming pool, the displacement is $(7.5 - 1.5, 10 - 2) = (6, 8)$: the distance is 10 km, from a).

c) From G to the station: $(-5, 3)$, and $\sqrt{25 + 9} = \sqrt{34} \approx 5.83$ km. From the station to the swimming pool: $(11, 5)$, and $\sqrt{121 + 25} = \sqrt{146} \approx 12.08$ km.

d) Via the station: $5.83 + 12.08 \approx 17.91$ km, that is, about 7.91 km more than the direct trip. A detour never shortens a trip.

e) On the ground, $\sqrt{4 + 9} = \sqrt{13}$, then $\sqrt{13 + 36} = \sqrt{49} = 7$. Directly: $\sqrt{4 + 9 + 36} = 7$.

In [ ]:
# Solution: the norm of a vector, without np.linalg.norm (equation 8.1)
def norm(u):
    u = np.asarray(u, dtype=float)
    return float(np.sqrt((u ** 2).sum()))

pool, station = np.array([7.5, 10]), np.array([-3.5, 5])   # swimming pool and station, in km

In [ ]:
# Check of a) to e)
if norm([6, 8]) is None:
    print('not yet completed. Complete the function norm.')
else:
    d_pool, d_station, d_station_pool = norm(pool - G_map), norm(station - G_map), norm(pool - station)
    print(f"a) {norm([6, 8]):g}   b) {d_pool:g} km   c) {d_station:.2f} km and {d_station_pool:.2f} km")
    print(f"d) detour: {d_station + d_station_pool - d_pool:.2f} km   e) {norm([2, 3, 6]):g}")
    print('→', 'correct' if abs(d_pool - 10) < 1e-9 and abs(d_station - 5.831) < 5e-4
          and abs(d_station_pool - 12.083) < 5e-4 and abs(norm([2, 3, 6]) - 7) < 1e-9 else 'check again')

## Exercise 8.2 — The neighbors of home I

*Concepts: raw distance and standardized distance (lesson 8.2, equation 8.2).* Home I (dataset D2.E2) has a `floor_area` of 75 m² and a `distance` of 3.5 km: it is the vector `I`. We estimate it with the six reference sales and $k = 3$.

- `floor_area` — floor area of the home, in m² (column `surface_m2` of the CSV);
- `distance` — distance to the center, in km (column `distance_km` of the CSV);
- `price` — prices of the reference sales, in k€; $z$-scores without a unit.

### a) to e) The calculation by hand

a) Differences (`floor_area`; `distance`) and raw distances to I: A $(-43, -0.5)$, 43.00; B $(-30, -2)$, 30.07; C $(-21, -1)$, 21.02; D $(-5, -2.5)$, 5.59; E $(10, -1)$, 10.05; F $(23, -3)$, 23.19. The neighbors are D, E and C: $(245 + 279 + 185)/3 \approx 236.3$ k€.

b) $z_s = (75 - 64)/22.80 \approx 0.48$; $z_d = (3.5 - 1.833)/0.898 \approx 1.86$.

c) Standardized distances: A 1.97; B 2.59; C 1.45; D 2.79; E 1.20; F 3.49. For E: $\sqrt{(0.92 - 0.48)^2 + (0.74 - 1.86)^2} = \sqrt{0.19 + 1.25} \approx 1.20$. The neighbors are E, C and A: $(279 + 185 + 118)/3 = 194$ k€.

d) D has disappeared. Its floor area is close to that of I (70 m² versus 75), but it is 1 km from the center, versus 3.5 km for I. In standard deviations, these 2.5 km weigh $2.5/0.898 \approx 2.79$, much more than the 5 m² ($5/22.80 \approx 0.22$). A, which is also far from the center, replaces it.

e) The nearest standardized neighbor, E, is already 1.20 standard deviations away from I: no reference sale really resembles it. An estimate from neighbors stays between the known prices; it is fragile for a home outside the area covered by the data.

In [ ]:
# Solution: distances from I to the six sales A to F, raw then standardized
raw_I = np.sqrt(((X - I) ** 2).sum(axis=1))
z_I = (I - mu) / sigma
std_I = np.linalg.norm(Z - z_I, axis=1)

In [ ]:
# Check of a) to c)
if raw_I is None or z_I is None or std_I is None:
    print('not yet completed. Complete the three lines.')
else:
    show_table(raw_I, 'raw_distance')
    (n_raw, e_raw), (n_std, e_std) = nearest(raw_I), nearest(std_I)
    print(f"a) raw: {n_raw}, estimate {e_raw:.1f} k€")
    print(f"b) z of I: {np.round(z_I, 2)}")
    show_table(std_I, 'std_distance')
    print(f"c) standardized: {n_std}, estimate {e_std:.0f} k€")
    print('→', 'correct' if n_raw == ['D', 'E', 'C'] and abs(e_raw - 236.33) < 0.01
          and n_std == ['E', 'C', 'A'] and abs(e_std - 194) < 1e-9 else 'check again')

## Exercise 8.3 — Meters or kilometers?

*Concepts: raw distance and units (lesson 8.2).* We go back to G and the six sales, but the distance to the center is counted in meters: `distance_m` $= 1{,}000 \times$ `distance`.

- `distance_m` — distance to the center, in m;
- `distance` — distance to the center, in km; `floor_area` (m²) and `price` (k€) of the reference sales.

### a) to d) The calculation by hand

a) Differences in distance: 500, −1,000, 0, −1,500, 0 and −2,000 m. Raw distances to G: A 500.9; B 1,000.1; C 8; D 1,500.0; E 23; F 2,000.3. Neighbors C, E and A: $(185 + 279 + 118)/3 = 194$ k€.

b) In kilometers, the floor area decided on its own; in meters, it is the distance that decides: the three neighbors are the three sales located 2.5 or 3 km away. The floor area now only serves to break the tie between C and E.

c) The mean becomes 1,833 m and the standard deviation 898 m: both are multiplied by 1,000. In $(x - \mu)/\sigma$, numerator and denominator are multiplied by 1,000: the score does not change. For G: $(2{,}500 - 1{,}833)/898 \approx 0.74$, as in kilometers.

d) The scores are identical, so the standardized distances are too: the neighbors remain C, E and B, and the estimate 212 k€. Standardization makes the neighbors independent of the units.

In [ ]:
# Solution: the distance to the center in meters, then the raw distances to G
distance_m = 1000 * reference['distance'].to_numpy(dtype=float)
G_distance_m = 1000 * G[1]
raw_G_m = np.sqrt((reference['floor_area'].to_numpy() - G[0]) ** 2 + (distance_m - G_distance_m) ** 2)

In [ ]:
# Check of a), c) and d)
if distance_m is None or G_distance_m is None or raw_G_m is None:
    print('not yet completed. Complete the three lines.')
else:
    show_table(raw_G_m, 'raw_distance_m')
    n_m, e_m = nearest(raw_G_m)
    print(f"a) in meters: {n_m}, estimate {e_m:.0f} k€")
    mu_m, sigma_m = distance_m.mean(), distance_m.std()
    z_G_m = (G_distance_m - mu_m) / sigma_m
    print(f"c) mean {mu_m:.0f} m, standard deviation {sigma_m:.0f} m; z of G: {z_G_m:.2f} in m, {z_G[1]:.2f} in km")
    print('→', 'correct' if n_m == ['C', 'E', 'A'] and abs(e_m - 194) < 1e-9 and abs(z_G_m - z_G[1]) < 1e-12
          else 'check again')

## Exercise 8.4 — How many neighbors?

*Concepts: nearest neighbors, choice of $k$ (lesson 8.2), validation (lesson 6.4).* In order of standardized distance to G, the six sales and their `price` are C (185 k€), E (279), B (172), A (118), D (245) and F (335).

- `price` — sale price, in k€; $k$ — number of neighbors kept, without a unit.

### a) to c) The calculation by hand

a) $k = 1$: 185 k€. $k = 2$: $(185 + 279)/2 = 232$ k€. $k = 3$: 212 k€. $k = 4$: $(636 + 118)/4 = 188.5$ k€. $k = 6$: $1{,}334/6 \approx 222.3$ k€.

b) It is the mean price of the six sales: the same estimate for every home. It completely ignores the characteristics of G.

c) $k = 3$ gives the smallest difference: +3 k€, versus −24, +23, −20.5 and +13.3 k€. But choosing $k$ based on G, the sale to predict, amounts to looking at the answer: it is the data leakage of chapter 6. You would need to choose $k$ on validation sales, for example by cross-validation (lesson 6.4), then judge the estimate on sales that were not used to decide anything.

In [ ]:
# Solution: the estimate for G for each value of k
estimates = {k: nearest(std_G, k)[1] for k in (1, 2, 3, 4, 6)}

In [ ]:
# Check of a) and c)
if estimates is None:
    print('not yet completed. Complete the line.')
else:
    for k, e in estimates.items():
        print(f"k = {k}: {e:.1f} k€, difference from the actual price (209 k€): {e - 209:+.1f} k€")
    expected = {1: 185, 2: 232, 3: 212, 4: 188.5, 6: 1334 / 6}
    print('→', 'correct' if all(abs(estimates[k] - v) < 1e-9 for k, v in expected.items()) else 'check again')

## Exercise 8.5 — Distance or cosine?

*Concepts: distance (equation 8.1), cosine (equation 8.3).* We count three words in four of the association's messages: “meeting”, “offer” and “volunteer” (table 8.4 in the book).

- `message` — message identifier, T1 to T4;
- `meeting_count`, `offer_count`, `volunteer_count` — number of times the words “meeting”, “offer” and “volunteer” are used in the message: the three coordinates of its vector.

In [ ]:
# Data for exercise 8.5 (table in the book)
words = pd.DataFrame({
    'message':         ['T1', 'T2', 'T3', 'T4'],
    'meeting_count':   [2, 6, 1, 0],
    'offer_count':     [0, 0, 2, 1],
    'volunteer_count': [1, 3, 2, 0],
}).set_index('message')
T = {m: words.loc[m].to_numpy(dtype=float) for m in words.index}
print(words)

### a) to d) The calculation by hand

a) $\lVert \text{T1} \rVert = \sqrt{4 + 0 + 1} = \sqrt{5} \approx 2.24$; $\lVert \text{T2} \rVert = \sqrt{36 + 9} = \sqrt{45} \approx 6.71$; $\lVert \text{T3} \rVert = \sqrt{1 + 4 + 4} = 3$; $\lVert \text{T4} \rVert = 1$.

b) T1 − T2 $= (-4, 0, -2)$, distance $\sqrt{20} \approx 4.47$. T1 − T3 $= (1, -2, -1)$, distance $\sqrt{6} \approx 2.45$. T1 − T4 $= (2, -1, 1)$, distance $\sqrt{6} \approx 2.45$. T3 and T4 are tied as the closest; T2 is the farthest.

c) T1 · T2 $= 15$ and $\cos = 15/(\sqrt{5} \times \sqrt{45}) = 15/15 = 1$. T1 · T3 $= 4$ and $\cos = 4/(\sqrt{5} \times 3) \approx 0.60$. T1 · T4 $= 0$ and $\cos = 0$. T2 is the most similar, then T3, then T4.

d) T2 has exactly the proportions of T1: the same topic, developed three times as much. The distance penalizes it for its length. T4 has no word in common with T1, but it is short: its tip is close to that of T1. To sort by topic, the cosine is the right choice: it compares the proportions of the words.

In [ ]:
# Solution: the cosine of two vectors (equation 8.3)
def cosine(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

In [ ]:
# Check of a) to c)
if cosine(T['T1'], T['T2']) is None:
    print('not yet completed. Complete the function cosine.')
else:
    results = pd.DataFrame({
        'norm': [np.linalg.norm(T[m]) for m in T],
        'distance_to_T1': [np.linalg.norm(T['T1'] - T[m]) for m in T],
        'cosine_with_T1': [cosine(T['T1'], T[m]) for m in T],
    }, index=list(T)).round(2)
    print(results)
    print('→', 'correct' if abs(cosine(T['T1'], T['T2']) - 1) < 1e-12 and abs(cosine(T['T1'], T['T3']) - 0.596) < 5e-4
          and abs(cosine(T['T1'], T['T4'])) < 1e-12 else 'check again')

## Exercise 8.6 — Challenge: normalize, then compare

*Concepts: norm and dot product (equation 8.1), cosine (equation 8.3).* Questions a) and b) are proofs on paper; the notebook checks c) and d).

### a) to d) The calculation by hand

a) With the rules of the dot product: $(\mathbf{u} - \mathbf{v}) \cdot (\mathbf{u} - \mathbf{v}) = \mathbf{u} \cdot \mathbf{u} - 2\, \mathbf{u} \cdot \mathbf{v} + \mathbf{v} \cdot \mathbf{v} = \lVert \mathbf{u} \rVert^2 + \lVert \mathbf{v} \rVert^2 - 2\, \mathbf{u} \cdot \mathbf{v}$.

b) If both norms equal 1, then $\cos(\mathbf{u}, \mathbf{v}) = \mathbf{u} \cdot \mathbf{v}$, and a) gives $1 + 1 - 2 \cos$.

c) Normalizing does not change the cosines. T2: $\sqrt{2 - 2 \times 1} = 0$; T3: $\sqrt{2 - 2 \times 0.596} \approx 0.90$; T4: $\sqrt{2 - 0} \approx 1.41$. The ranking is that of the cosine: the larger the cosine, the smaller the distance.

d) $(4, 3)/5 = (0.8, 0.6)$ and $(3, 4)/5 = (0.6, 0.8)$. Directly, the difference is $(0.2, -0.2)$ and the distance $\sqrt{0.08} \approx 0.28$. With b): $\sqrt{2 - 2 \times 0.96} = \sqrt{0.08} \approx 0.28$.

In [ ]:
# Solution: each message divided by its norm, then the distances to the normalized T1
normalized = {m: T[m] / np.linalg.norm(T[m]) for m in T}
unit_dist = {m: float(np.linalg.norm(normalized['T1'] - normalized[m])) for m in T}

In [ ]:
# Check of c) and d)
if normalized is None or unit_dist is None:
    print('not yet completed. Complete the two lines.')
else:
    for m in ('T2', 'T3', 'T4'):
        c = float(np.dot(T['T1'], T[m]) / (np.linalg.norm(T['T1']) * np.linalg.norm(T[m])))
        print(f"{m}: normalized distance {unit_dist[m]:.2f}, √(2 − 2 cos) = {np.sqrt(max(2 - 2 * c, 0)):.2f}")
    u, v = embeddings['meeting'] / 5, embeddings['assembly'] / 5
    print(f"d) meeting and assembly: directly {np.linalg.norm(u - v):.2f}, with b) {np.sqrt(2 - 2 * np.dot(u, v)):.2f}")
    print('→', 'correct' if abs(unit_dist['T2']) < 1e-9 and abs(unit_dist['T3'] - 0.899) < 5e-4
          and abs(unit_dist['T4'] - np.sqrt(2)) < 1e-9 else 'check again')